# Iniciando o projeto (Scaffold)

### Criar a arquitetura de diretórios

`agentcore create --name CustomerSupport --framework Strands --model-provider Bedrock --memory none`

`cd CustomerSupport`

### Estrutura de pastas

```text
CustomerSupport/
├── AGENTS.md                          # AI assistant context file
├── README.md
├── agentcore/
│   ├── agentcore.json                 # Main project config
│   ├── aws-targets.json               # Deployment targets
│   ├── .env.local                     # API keys (gitignored)
│   ├── .cli/deployed-state.json       # Deployment state (auto-managed)
│   ├── .llm-context/                  # TypeScript type definitions
│   └── cdk/                           # CDK infrastructure
└── app/
    └── CustomerSupport/
        ├── main.py                    # Agent entry point
        ├── model/load.py              # Model configuration
        ├── mcp_client/client.py       # MCP client (Exa AI web search)
        └── pyproject.toml             # Python dependencies
```

# Customizando o agente

### Substituir o padrão do app/main.py 

```python
from strands import Agent, tool
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from model.load import load_model
from mcp_client.client import get_streamable_http_mcp_client

app = BedrockAgentCoreApp()
log = app.logger

# Define a Streamable HTTP MCP Client
mcp_clients = [get_streamable_http_mcp_client()]

# Resumos dos livros (chaves = títulos em português)
RESUMOS_LIVROS = {
    "Cinquenta Tons de Cinza": (
        "Anastasia Steele, estudante de literatura, entrevista o jovem bilionário "
        "Christian Grey para o jornal da faculdade. Os dois se envolvem em um "
        "relacionamento intenso, marcado pelos segredos e pelo passado conturbado "
        "de Grey, que obriga Ana a decidir até onde está disposta a ir."
    ),
    "Harry Potter e a Pedra Filosofal": (
        "Harry, um órfão criado pelos tios, descobre aos 11 anos que é bruxo e vai "
        "estudar em Hogwarts. Lá faz amizade com Rony e Hermione e descobre que "
        "alguém pretende roubar a Pedra Filosofal para trazer de volta Voldemort, "
        "o bruxo que matou seus pais."
    ),
    "Código Limpo": (
        "Guia prático sobre como escrever código legível e fácil de manter. "
        "Aborda nomes significativos, funções pequenas, comentários, formatação, "
        "tratamento de erros, testes unitários e refatoração, com vários "
        "estudos de caso em Java."
    ),
    "Gash Bell": (
        "Kiyomaro Takamine, um estudante genial e solitário, passa a cuidar de "
        "Gash, um menino-demônio sem memória. Gash participa de uma batalha "
        "entre 100 demônios para decidir o próximo rei do mundo demoníaco, e cada "
        "um precisa de um parceiro humano que leia o livro de feitiços."
    ),
    "O Hobbit": (
        "Bilbo Bolseiro, um hobbit pacato, é recrutado pelo mago Gandalf e por "
        "treze anões para recuperar o tesouro tomado pelo dragão Smaug. Na jornada "
        "enfrenta trolls, orcs e aranhas gigantes, e encontra o Um Anel."
    ),
}
 
# Informações dos livros
# Número de páginas: edição original (varia conforme a edição/tradução).
INFOS_LIVROS = {
    "Cinquenta Tons de Cinza": {"id": 1, "numero_paginas": 514, "autor": "E. L. James"},
    "Harry Potter e a Pedra Filosofal": {"id": 2, "numero_paginas": 223, "autor": "J. K. Rowling"},
    "Código Limpo": {"id": 3, "numero_paginas": 464, "autor": "Robert C. Martin"},
    # Mangá em 33 volumes: não há um número único de páginas
    "Gash Bell": {"id": 4, "numero_paginas": None, "autor": "Makoto Raiku"},
    "O Hobbit": {"id": 5, "numero_paginas": 310, "autor": "J. R. R. Tolkien"},
}

@tool
def obter_resumo_livro(nome_livro: str) -> str:
    """Obtém o resumo de um livro a partir do seu título em português.
 
    Args:
        nome_livro: título do livro, por exemplo "O Hobbit".
    """
    # Busca sem diferenciar maiúsculas/minúsculas e ignorando espaços nas pontas
    chave = nome_livro.strip().lower()
    for titulo, resumo in RESUMOS_LIVROS.items():
        if titulo.lower() == chave:
            return resumo or "Não existe resumo para este livro ainda"
    return "Não existe resumo para este livro ainda"


@tool
def obter_informacoes_livro(nome_livro: str) -> str:
    """Obtém as informações (id, número de paginas e nome do autor) de um livro a partir do seu título em português.
 
    Args:
        nome_livro: título do livro, por exemplo "O Hobbit".
    """
    # Busca sem diferenciar maiúsculas/minúsculas e ignorando espaços nas pontas
    chave = nome_livro.strip().lower()
    for titulo, infos in INFOS_LIVROS.items():
        if titulo.lower() == chave:
            return infos or "Não existe resumo para este livro ainda"
    return "Não existe resumo para este livro ainda"

tools = [obter_resumo_livro, obter_informacoes_livro]

# Add MCP client to tools if available
for mcp_client in mcp_clients:
    if mcp_client:
        tools.append(mcp_client)


_agent = None

def get_or_create_agent():
    global _agent
    if _agent is None:
        _agent = Agent(
            model=load_model(),
            system_prompt="""
                Você é uma bibliotecária útil.

                Seu papel é Dar informações precisas aos clientes sobre os livros da biblioteca. 
            
                Você tem acesso à tools que te ajudam a encontrar resumos dos livros para ajudar na indicação e também outra que possui informações básicas sobre estes livros.
            """,
            tools=tools,
            callback_handler=None
        )
    return _agent

@app.entrypoint
async def invoke(payload, context):
    log.info("Invoking Agent.....")

    agent = get_or_create_agent()

    stream = agent.stream_async(payload.get("prompt"))
    async for event in stream:
        if "data" in event and isinstance(event["data"], str):
            yield event["data"]


if __name__ == "__main__":
    app.run()

```

### Substitua o app/CustomerSupport/model/load.py por:

```python
from strands.models.bedrock import BedrockModel

def load_model() -> BedrockModel:
    """Get Bedrock model client using IAM credentials."""
    return BedrockModel(model_id="global.anthropic.claude-sonnet-4-6")
```

### Arquitetura até aqui

![Descrição da imagem](imagens\lab1_architecture_diagram.png)

# Startando um servidor local (Dev)

```agentcore dev --no-browser```

o "--no-browser" serve basicamente para podermos interagir com o modelo pelo terminal, sem ele o agentcore tenta abrir um browser para interagir com o modelo via interface gráfica.

# Deploy do agente

```agentcore deploy```

```agentcore status```

Agora o agente já está na nuvem, pode checar o Agentcore runtime no console da AWS que ele estará lá

# Invocando o Agente

```agentcore invoke "What can you do?"```

# Complementos

- https://www.youtube.com/watch?v=usFIb9aEd1U